# Lab 02-07 · Determinants, Inverses, Rank and Vector Spaces

Companion to chapter [02-07](../../chapters/02-07-determinants-inverses-and-vector-spaces.md). Handout: [README.md](README.md).

Run the cells in order. Cells marked **🔮 Predict** ask you to write a prediction *before* running.

In [1]:
import math, sys, time
from pathlib import Path
import numpy as np
import scipy.linalg as sl
import sympy as sp
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd().parents[3] / "tools"))
try:
    from plotstyle import use_style; use_style()
except ImportError:
    pass
np.set_printoptions(precision=4, suppress=True)

def best_time(f, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(); ts.append(time.perf_counter() - t0)
    return min(ts)

## Part 1 — Determinants as volume, and what they cost

Map the corners of the unit square through a matrix and measure the area of the image with the shoelace formula. Compare with $\lvert\det\mathbf{A}\rvert$.

In [2]:
def shoelace(pts):
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

square = np.array([[0, 0], [1, 0], [1, 1], [0, 1.0]])
for A in (np.array([[3, 1], [2, 4.0]]), np.array([[1, 2], [3, 4.0]]), np.array([[2, 1], [4, 2.0]])):
    img = square @ A.T
    print(f"A = {A.tolist()}: image area = {shoelace(img):.4f}, det = {np.linalg.det(A):+.4f}")

A = [[3.0, 1.0], [2.0, 4.0]]: image area = 10.0000, det = +10.0000
A = [[1.0, 2.0], [3.0, 4.0]]: image area = 2.0000, det = -2.0000
A = [[2.0, 1.0], [4.0, 2.0]]: image area = 0.0000, det = +0.0000


A recursive cofactor expansion does $n$ smaller expansions of size $n - 1$, about $n!$ work in total.

**🔮 Predict:** if $n = 9$ takes $t$ seconds by cofactors, how long would $n = 12$ take?

In [3]:
def det_cofactor(M):
    n = len(M)
    if n == 1:
        return M[0][0]
    return sum((-1) ** j * M[0][j] * det_cofactor([row[:j] + row[j + 1:] for row in M[1:]]) for j in range(n))

rng = np.random.default_rng(0)
for n in range(5, 10):
    M = rng.standard_normal((n, n))
    t_cof = best_time(lambda: det_cofactor(M.tolist()), reps=1)
    t_lu = best_time(lambda: np.linalg.det(M))
    print(f"n = {n}: cofactors {t_cof*1e3:9.2f} ms   LU {t_lu*1e6:7.1f} us   "
          f"agree: {np.isclose(det_cofactor(M.tolist()), np.linalg.det(M))}")
print("predicted ratio 12!/9! =", math.factorial(12) // math.factorial(9))

n = 5: cofactors      0.13 ms   LU     3.8 us   agree: True
n = 6: cofactors      0.80 ms   LU     3.6 us   agree: True
n = 7: cofactors      4.92 ms   LU     3.6 us   agree: True
n = 8: cofactors     40.16 ms   LU     5.2 us   agree: True


n = 9: cofactors    396.05 ms   LU     4.0 us   agree: True
predicted ratio 12!/9! = 1320


## Part 2 — Inverse versus solve

**🔮 Predict:** for a 1000×1000 system, is `inv(A) @ b` faster or slower than `solve(A, b)`, and by roughly how much? Which is more accurate?

In [4]:
n = 1000
A = rng.standard_normal((n, n)); b = rng.standard_normal(n)
t_inv = best_time(lambda: np.linalg.inv(A) @ b)
t_sol = best_time(lambda: np.linalg.solve(A, b))
x_inv = np.linalg.inv(A) @ b; x_sol = np.linalg.solve(A, b)
print(f"inv(A) @ b : {t_inv*1e3:7.1f} ms, residual |Ax - b| = {np.linalg.norm(A @ x_inv - b):.2e}")
print(f"solve(A, b): {t_sol*1e3:7.1f} ms, residual |Ax - b| = {np.linalg.norm(A @ x_sol - b):.2e}")
print(f"time ratio: {t_inv / t_sol:.1f}  (theory: about 2n^3 vs (2/3)n^3 flops, a factor of 3)")

inv(A) @ b :    34.7 ms, residual |Ax - b| = 4.88e-11
solve(A, b):     9.5 ms, residual |Ax - b| = 3.36e-11
time ratio: 3.6  (theory: about 2n^3 vs (2/3)n^3 flops, a factor of 3)


## Part 3 — Rank, column space and null space

Exact arithmetic first, with SymPy, on Example 02-07.3.

In [5]:
A3 = sp.Matrix([[1, 2, 0, 1], [2, 4, 1, 4], [3, 6, 2, 7]])
R, pivots = A3.rref()
print("rref =", R.tolist(), " pivot columns:", pivots, " rank:", A3.rank())
print("null-space basis:", [list(v) for v in A3.nullspace()])
print("column-space basis:", [list(v) for v in A3.columnspace()])
b1, b2, b3 = sp.symbols("b1 b2 b3")
# carry b through the same elimination steps: R2 - 2R1, R3 - 3R1, then R3 - 2R2
r2 = b2 - 2 * b1; r3 = b3 - 3 * b1; r3 = r3 - 2 * r2
print("solvability condition:", sp.expand(r3), "= 0")

rref = [[1, 2, 0, 1], [0, 0, 1, 2], [0, 0, 0, 0]]  pivot columns: (0, 2)  rank: 2
null-space basis: [[-2, 1, 0, 0], [-1, 0, -2, 1]]
column-space basis: [[1, 2, 3], [0, 1, 2]]
solvability condition: b1 - 2*b2 + b3 = 0


In floating point, rank is a judgment call. A matrix that is rank 2 in exact arithmetic picks up tiny nonzero values from roundoff or measurement noise. NumPy's `matrix_rank` counts the singular values above a tolerance (singular values arrive in 02-08).

**🔮 Predict:** a rank-2 matrix gets noise of size $10^{-13}$ added. What rank will NumPy report with its default tolerance?

In [6]:
u1, u2, v1, v2 = (rng.standard_normal(6) for _ in range(4))
M = np.outer(u1, v1) + np.outer(u2, v2)          # exactly rank 2 in exact arithmetic
M_noisy = M + 1e-13 * rng.standard_normal(M.shape)
sv = np.linalg.svd(M_noisy, compute_uv=False)
print("singular values:", ", ".join(f"{s:.2e}" for s in sv))
tol_default = sv.max() * max(M_noisy.shape) * np.finfo(float).eps
print(f"NumPy's default tolerance here: {tol_default:.2e}; noise singular values below it: {int(np.sum(sv[2:] < tol_default))} of {sv.size - 2}")
print("rank with NumPy's default tolerance:", np.linalg.matrix_rank(M_noisy))
print("rank with a tolerance of 1e-10 (matched to the data's accuracy):", np.linalg.matrix_rank(M_noisy, tol=1e-10))
print("orthonormal null-space basis with that tolerance has", sl.null_space(M_noisy, rcond=1e-10).shape[1],
      "vectors (6 - 2 = 4)")

singular values: 8.52e+00, 5.20e+00, 2.65e-13, 1.85e-13, 8.99e-14, 1.05e-14
NumPy's default tolerance here: 1.13e-14; noise singular values below it: 1 of 4
rank with NumPy's default tolerance: 5
rank with a tolerance of 1e-10 (matched to the data's accuracy): 2
orthonormal null-space basis with that tolerance has 4 vectors (6 - 2 = 4)


The default tolerance assumes the only errors are double-precision roundoff (about $10^{-14}$ here), so noise at $10^{-13}$ counts as real: rank 5. A tolerance matched to the data's actual accuracy recovers rank 2. Numerical rank is always "rank at a stated tolerance".

## Part 4 — Least squares

Example 02-07.4 three ways. The residual must be orthogonal to both columns of $\mathbf{A}$.

In [7]:
T = np.array([0, 20, 40, 60, 80.0]); V = np.array([0.402, 0.598, 0.801, 0.997, 1.205])
A = np.c_[np.ones_like(T), T]
x_ne = np.linalg.solve(A.T @ A, A.T @ V)             # normal equations
x_ls = np.linalg.lstsq(A, V, rcond=None)[0]           # SVD-based library routine
x_pf = np.polyfit(T, V, 1)[::-1]                      # polyfit returns the highest power first
e = V - A @ x_ls
for name, xx in (("normal equations", x_ne), ("lstsq", x_ls), ("polyfit", x_pf)):
    print(f"{name:16s}: offset c = {xx[0]:.4f} V, sensitivity d = {xx[1]*1e3:.3f} mV/°C")
print("residuals (mV):", e * 1e3, "  A^T e =", A.T @ e)

normal equations: offset c = 0.3996 V, sensitivity d = 10.025 mV/°C
lstsq           : offset c = 0.3996 V, sensitivity d = 10.025 mV/°C
polyfit         : offset c = 0.3996 V, sensitivity d = 10.025 mV/°C
residuals (mV): [ 2.4 -2.1  0.4 -4.1  3.4]   A^T e = [-0. -0.]


**🔮 Predict:** fit polynomials of degree 1 to 4 to the same five points. What happens to the sum of squared residuals at degree 4, and what happens to the coefficients?

In [8]:
for deg in range(1, 5):
    c = np.polyfit(T, V, deg)
    sse = np.sum((np.polyval(c, T) - V) ** 2)
    print(f"degree {deg}: SSE = {sse:.3e} V^2   coefficients (highest power first): " + ", ".join(f"{ci:.3e}" for ci in c))

degree 1: SSE = 3.870e-05 V^2   coefficients (highest power first): 1.002e-02, 3.996e-01
degree 2: SSE = 1.806e-05 V^2   coefficients (highest power first): 3.036e-06, 9.782e-03, 4.020e-01
degree 3: SSE = 1.556e-05 V^2   coefficients (highest power first): 5.208e-08, -3.214e-06, 9.961e-03, 4.015e-01
degree 4: SSE = 5.448e-30 V^2   coefficients (highest power first): 8.594e-09, -1.323e-06, 6.406e-05, 8.979e-03, 4.020e-01


At degree 4 the polynomial passes through every point: zero residual, but it has fitted the noise too. Five parameters from five points leave nothing to estimate the noise with, and the curvature terms are artifacts. More parameters are not better.

## Part 5 — Circuits as graphs

The incidence matrix of Example 02-07.5 (nodes 0–3; branches a = 1→2, b = 1→3, c = 2→0, d = 2→3, e = 3→0).

In [9]:
branches = [(1, 2), (1, 3), (2, 0), (2, 3), (3, 0)]
g = np.array([1, 0.5, 1, 0.5, 2.0])                    # mS
A = np.zeros((5, 4))
for k, (s, t) in enumerate(branches):
    A[k, s], A[k, t] = 1, -1
print("incidence matrix:\n", A)
print("rank:", np.linalg.matrix_rank(A), "  independent loops:", 5 - np.linalg.matrix_rank(A))
loops = sp.Matrix(A.astype(int)).T.nullspace()          # N(A^T): branch-current patterns with no node injection
print("loop basis (branch currents a..e):", [list(v) for v in loops])
Ar = A[:, 1:]                                          # ground node 0 removed
G = Ar.T @ np.diag(g) @ Ar
print("G = Ar^T C Ar (mS):\n", G)
R = np.linalg.inv(G)
print("R = G^-1 (kOhm):\n", R)
print("symmetric (reciprocity):", np.allclose(R, R.T), "  10 mA at node 1 ->", R[:, 0] * 10, "V")

incidence matrix:
 [[ 0.  1. -1.  0.]
 [ 0.  1.  0. -1.]
 [-1.  0.  1.  0.]
 [ 0.  0.  1. -1.]
 [-1.  0.  0.  1.]]
rank: 3   independent loops: 2
loop basis (branch currents a..e): [[1, -1, 0, 1, 0], [-1, 1, -1, 0, 1]]
G = Ar^T C Ar (mS):
 [[ 1.5 -1.  -0.5]
 [-1.   2.5 -0.5]
 [-0.5 -0.5  3. ]]
R = G^-1 (kOhm):
 [[1.0741 0.4815 0.2593]
 [0.4815 0.6296 0.1852]
 [0.2593 0.1852 0.4074]]
symmetric (reciprocity): True   10 mA at node 1 -> [10.7407  4.8148  2.5926] V


## Part 6 — ngspice

From the repository root, run `python tools/run_spice.py 02-07`. The netlist `resistance_matrix.cir` injects a unit current at one node at a time, and reads off columns of the resistance matrices:

- the ladder of Example 02-07.2 (0.75, 0.5, 0.25 Ω and 1 Ω);
- the circuit of Example 02-07.5 (0.4815, 0.6296 and 0.1852 kΩ from node 2; 0.2593 and 0.4074 kΩ from node 3, reciprocity included);
- Problems 02-07.11 to 02-07.13.

## Your turn

1. Add a sixth branch from node 1 to ground in Part 5. Predict the new rank and the new number of loops, then check.
2. In Part 4, center the temperatures ($T - 40$) before fitting a cubic. Compare the coefficients and the condition number of $\mathbf{A}^\mathsf{T}\mathbf{A}$ (`np.linalg.cond`) with and without centering.
3. In Part 2, replace the random matrix with the 1000×1000 matrix $0.1\,\mathbf{I}$. Compute its determinant. Is it "nearly singular"?